In [22]:
!pip uninstall -y torchao
!kill 15512

/bin/bash: line 1: kill: (15512) - No such process


In [23]:
!pip install -q \
transformers \
datasets \
accelerate \
peft \
sentencepiece \
protobuf \
safetensors

In [24]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [25]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

True
Tesla T4
14.56317138671875 GB


In [26]:
!git clone https://github.com/ToastCoder/pr-assist.git
%cd pr-assist

fatal: destination path 'pr-assist' already exists and is not an empty directory.
/content/pr-assist/pr-assist


In [27]:
import torch
import transformers
import datasets
import peft

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

Torch: 2.11.0+cu128
Transformers: 5.12.1
Datasets: 5.0.0
PEFT: 0.19.1


In [28]:
import os

print(os.getcwd())
DATASET_NAME = "opencsg/PR_review_deepseek"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = "./pr-assist-model"

/content/pr-assist/pr-assist


In [29]:
from datasets import load_dataset

dataset = load_dataset(
    DATASET_NAME,
    split="train"
)

print(f"Samples: {len(dataset)}")
print(dataset[0])

Samples: 24821
{'instruction': 'Here is a python code snippet (Some lines may be omitted):\n\n```python\n\nimport sys\nimport threading\nimport signal\nimport array\nimport queue\nimport time\nimport os\nfrom os import getpid\n\nfrom traceback import format_exc\n\nfrom . import connection\nfrom .context import reduction, get_spawning_popen, ProcessError\nfrom . import pool\nfrom . import process\nfrom . import util\nfrom . import get_context\ntry:\n    from . import shared_memory\n    HAS_SHMEM = True\n    __all__.append(SharedMemoryManager)\nexcept ImportError:\n    HAS_SHMEM = False\n\n#\n# Register some things for pickling\n#\n\ndef reduce_array(a):\n    return array.array, (a.typecode, a.tobytes())\nreduction.register(array.array, reduce_array)\n\nview_types = [type(getattr({}, name)()) for name in (\'items\',\'keys\',\'values\')]\nif view_types[0] is not list:       # only needed in Py3.0\n    def rebuild_as_list(obj):\n        return list, (list(obj),)\n    for view_type in view_

In [30]:
MAX_LENGTH = 1024
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LR = 2e-4

In [31]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token

In [32]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [33]:
import peft
import transformers
import torch

print(peft.__version__)
print(transformers.__version__)
print(torch.__version__)

from peft import LoraConfig
from peft import get_peft_model

0.19.1
5.12.1
2.11.0+cu128


In [34]:
from peft import (
    LoraConfig,
    get_peft_model
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [35]:
def format_sample(sample):
    messages = [
        {
            "role": "user",
            "content": sample["instruction"]
        },
        {
            "role": "assistant",
            "content": sample["output"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

In [36]:
MAX_LENGTH = 2048


def tokenize_sample(sample):
    text = format_sample(sample)

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [37]:
tokenized_dataset = dataset.map(
    tokenize_sample,
    remove_columns=dataset.column_names
)

tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 24821
})

In [38]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    fp16=True
)

In [39]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [40]:
trainer.train()

Step,Training Loss
25,1.080816
50,1.099785
75,1.019932


KeyboardInterrupt: 

In [ ]:
trainer.save_model(
    f"{OUTPUT_DIR}/final"
)

tokenizer.save_pretrained(
    f"{OUTPUT_DIR}/final"
)

In [ ]:
from google.colab import files

files.download(
    f"{OUTPUT_DIR}/final/adapter_model.safetensors"
)

In [ ]:
files.download(
    f"{OUTPUT_DIR}/final/adapter_config.json"
)